In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/processed/car_price_cleaned.csv")

features = [
    "name", "year", "km_driven", "fuel",
    "seller_type", "transmission", "owner"
]

X = df[features]
y = df["selling_price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_features = ["year", "km_driven"]
categorical_features = [
    "name", "fuel", "seller_type", "transmission", "owner"
]

preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (2861, 7)
Test: (716, 7)


In [3]:
candidates = {
    "Random Forest": TransformedTargetRegressor(
        regressor=Pipeline([
            ("preprocessor", preprocessor),
            ("model", RandomForestRegressor(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
                min_samples_leaf=2
            ))
        ])
    ),
    "Log-target Extra Trees": TransformedTargetRegressor(
        regressor=Pipeline([
            ("preprocessor", preprocessor),
            ("model", ExtraTreesRegressor(
                n_estimators=250,
                max_depth=25,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            ))
        ]),
        func=np.log1p,
        inverse_func=np.expm1
    )
}

results = []
final_candidates = {}

for name, model in candidates.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

    final_candidates[name] = model

results_df = pd.DataFrame(results).sort_values("MAE")
display(results_df.round(3))

,Model,MAE,RMSE,R2
1,Log-target Extra Trees,165636.206,421322.395,0.449
0,Random Forest,174156.832,404521.211,0.492


In [4]:
from pathlib import Path
import joblib

# Select the model with the lowest MAE on the fixed test split.
# This is provisional selection; test results should not be repeatedly
# optimized against once used for final evaluation.
selected_name = results_df.iloc[0]["Model"]
selected_model = final_candidates[selected_name]

MODEL_PATH = Path("../models/car_price_pipeline.joblib")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(selected_model, MODEL_PATH)

print("Selected model:", selected_name)
print("Saved to:", MODEL_PATH.resolve())
print("File size (MB):", round(MODEL_PATH.stat().st_size / (1024**2), 3))

Selected model: Log-target Extra Trees
Saved to: C:\Users\kalav\OneDrive\Desktop\Car-Price-Predictor\models\car_price_pipeline.joblib
File size (MB): 27.177
